# 16 · Does reading the whole essay help? (windowed scoring, inference only)

**Goal.** The detector reads only the first 512 tokens (about 380 words) of an essay. Unseen-prompt human essays average 501 words, so many are cut off, and notebook 14 said the wrongly flagged essays were the longer ones. **Hypothesis: truncation contributes to false flags.** Notebook 15 weakened the premise (the "longer" gap is mostly the long Summer-projects prompt), so this is a clean, time-boxed test, and a negative result is a fine result. Nothing is trained or tuned beyond choosing one scoring variant on `val`.

**Rule written before looking at any held-out number.** The variant is chosen on `val` only, by the highest share of val AI essays caught at the threshold that flags 1% of val humans (ties: the higher val AUC). `test` and the held-out prompts are only read afterwards, to report. Summer projects and Electoral college are not used to choose anything.

1. A windowed scorer (`Detector.score_windows` in `src/detector.py`; `Detector.score` is untouched).
2. Choose the variant on val.
3. Compare with first-512 scoring: false flags by essay length, with intervals, and the extra time.
4. Say plainly whether the hypothesis held.

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')

import time
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score

from src.data import load_splits
from src.detector import Detector
from src.evaluate import wilson_interval, threshold_for_fpr, auc_with_interval

SEED, TARGET = 42, 0.01
pd.set_option('display.width', 220)

data = load_splits()
base = pd.read_csv('../results/15_scores.csv', index_col='row_id')       # first-512 DistilRoBERTa scores from 15
ev = data.loc[base.index].copy()
ev['first512'] = base['roberta']
print(ev['split'].value_counts().to_string())

split
heldout_prompt       7135
test                 3501
val                  3501
heldout_generator    2721


## 1 · The windowed scorer

The model sees at most 512 tokens: 510 of text plus a start and an end marker. `score_windows` tokenises the whole essay once, without cutting, and reads it in windows of 510 text tokens whose starts are `stride` tokens apart; the last window ends where the essay ends. Overlap means a sentence near a window edge is also seen in the middle of the next window. An essay of at most 510 tokens gets exactly one window, identical to what `Detector.score` feeds the model, so its score cannot change. Only longer essays can differ.

Per essay we get one logit per window and combine them three ways:

- **mean**: the average window logit;
- **max**: the most AI-looking window (a single suspicious stretch is enough to flag; expected to raise false flags);
- **length-weighted mean**: windows weighted by their token count, so a short tail window counts less.

Strides tried: 256 (windows overlap by half) and 384 (overlap by a quarter). Scoring all 16,858 evaluation essays takes a few minutes; results are cached in `results/16_window_scores.csv`.

In [2]:
CACHE = '../results/16_window_scores.csv'
if os.path.exists(CACHE):
    win = pd.read_csv(CACHE, index_col='row_id')
    timing = pd.read_csv('../results/16_timing.csv', index_col=0)['seconds']
    print('loaded', CACHE)
else:
    det = Detector()
    texts = ev['text'].tolist()
    win = pd.DataFrame(index=ev.index)
    win['n_tokens'] = [len(x) for x in det.tok(texts, add_special_tokens=False)['input_ids']]
    timing = {}
    t0 = time.time(); det.score(texts); timing['first-512 (Detector.score)'] = time.time() - t0
    for stride in (256, 384):
        t0 = time.time()
        logits, lengths = det.score_windows(texts, stride=stride)
        timing[f'windows, stride {stride}'] = time.time() - t0
        win[f'n_windows_{stride}'] = [len(x) for x in logits]
        win[f'w{stride}_mean'] = [x.mean() for x in logits]
        win[f'w{stride}_max'] = [x.max() for x in logits]
        win[f'w{stride}_lw'] = [(x * n).sum() / n.sum() for x, n in zip(logits, lengths)]
    timing = pd.Series(timing)
    win.to_csv(CACHE); timing.rename('seconds').to_frame().to_csv('../results/16_timing.csv')
    del det
ev = ev.join(win)
VARIANTS = ['first512'] + [f'w{s}_{a}' for s in (256, 384) for a in ('mean', 'max', 'lw')]
timing

loaded ../results/16_window_scores.csv


first-512 (Detector.score)     80.588906
windows, stride 256           104.586804
windows, stride 384            94.029738
Name: seconds, dtype: float64

### Sanity checks

An essay of at most 510 tokens has one window, so every windowed variant must give the same score as first-512 (up to fp16 rounding).

In [3]:
one = ev[ev['n_windows_256'] == 1]
for v in VARIANTS[1:]:
    print(f"{v:10s} max |difference from first-512| on the {len(one):,} single-window essays: {np.abs(one[v] - one['first512']).max():.4f}")
assert all(np.abs(one[v] - one['first512']).max() < 0.05 for v in VARIANTS[1:])
print(f"\nessays longer than 510 tokens (cut by first-512 scoring): {(ev['n_tokens'] > 510).mean():.1%} of all")
ev['bucket'] = pd.cut(ev['n_tokens'], [0, 510, 768, 1024, 10_000], labels=['<=510 tokens', '511-768', '769-1024', '>1024'])
(ev.groupby(['split', 'label', 'bucket'], observed=True).size().unstack('bucket').fillna(0).astype(int)
   .rename(index={0: 'human', 1: 'AI'}, level='label'))

w256_mean  max |difference from first-512| on the 11,833 single-window essays: 0.0176
w256_max   max |difference from first-512| on the 11,833 single-window essays: 0.0176
w256_lw    max |difference from first-512| on the 11,833 single-window essays: 0.0176
w384_mean  max |difference from first-512| on the 11,833 single-window essays: 0.0132
w384_max   max |difference from first-512| on the 11,833 single-window essays: 0.0132
w384_lw    max |difference from first-512| on the 11,833 single-window essays: 0.0132

essays longer than 510 tokens (cut by first-512 scoring): 29.8% of all


bucket                   <=510 tokens  511-768  769-1024  >1024
split             label                                        
heldout_generator AI             2704       17         0      0
heldout_prompt    human          1880     1708       662    214
                  AI             2118      429       124      0
test              human          1436      622       178     55
                  AI             1090      114         6      0
val               human          1508      603       129     51
                  AI             1097      108         5      0

## 2 · Choosing the variant on val

For each variant the threshold is the score that flags 1% of the **val humans** (the same rule as notebook 09). Then, on val only: the share of val AI essays caught, the AUC, and the false-flag rate among long val humans (those a variant can actually change). We pick the variant with the highest val AI-caught share (ties: higher val AUC), as declared at the top.

In [4]:
val = ev[ev['split'] == 'val']
vh = val[val['label'] == 0]
rows = []
for v in VARIANTS:
    thr = threshold_for_fpr(vh[v], TARGET)
    long_h = vh[vh['n_tokens'] > 510]
    rows.append({'variant': v, 'threshold': thr,
                 'val AI caught %': 100 * (val.loc[val['label'] == 1, v] >= thr).mean(),
                 'val AUC': roc_auc_score(val['label'], val[v]),
                 'long val humans flagged %': 100 * (long_h[v] >= thr).mean(), 'n long humans': len(long_h),
                 'short val humans flagged %': 100 * (vh.loc[vh['n_tokens'] <= 510, v] >= thr).mean()})
sel = pd.DataFrame(rows).set_index('variant')
chosen = sel.sort_values(['val AI caught %', 'val AUC'], ascending=False).index[0]
print('chosen on val by the declared rule:', chosen)
sel.round(4)

chosen on val by the declared rule: w384_lw


,threshold,val AI caught %,val AUC,long val humans flagged %,n long humans,short val humans flagged %
variant,,,,,,
first512,4.6660,99.4215,0.9997,1.0217,783,0.9947
w256_mean,4.5293,99.4215,0.9997,0.7663,783,1.1273
w256_max,8.0336,99.2562,0.9995,2.0434,783,0.4642
w256_lw,4.1512,99.4215,0.9997,0.6386,783,1.1936
w384_mean,4.6082,99.5041,0.9996,0.8940,783,1.0610
w384_max,8.6984,99.1736,0.9994,2.1711,783,0.3979
w384_lw,4.4266,99.5041,0.9996,0.7663,783,1.1273


## 3 · First-512 against the windowed variant: false flags by essay length

Now `test` and the held-out prompts are read, once. Thresholds are the val ones from the table above (1% of val humans, per variant). For each length bucket the tables give the share of **human** essays flagged with a Wilson interval, then the share of AI essays caught. All variants are shown in the second table, not only the chosen one, so nothing is hidden by the selection.

In [5]:
THR = {v: threshold_for_fpr(vh[v], TARGET) for v in VARIANTS}
def cell(k, n):
    lo, hi = wilson_interval(k, n)
    return f"{100 * k / n:.1f} [{100 * lo:.1f}, {100 * hi:.1f}] ({k}/{n})" if n else '-'

def by_bucket(split, label, variants):
    d = ev[(ev['split'] == split) & (ev['label'] == label)]
    out = {}
    for v in variants:
        flagged = d[v] >= THR[v]
        out[v] = {b: cell(int(flagged[d['bucket'] == b].sum()), int((d['bucket'] == b).sum())) for b in d['bucket'].cat.categories}
        out[v]['ALL'] = cell(int(flagged.sum()), len(d))
    return pd.DataFrame(out).T

show = ['first512', chosen]
for split in ('test', 'heldout_prompt'):
    print(f'HUMANS FLAGGED %, {split}  (threshold = val 1%)'); print(by_bucket(split, 0, show).to_string(), '\n')

HUMANS FLAGGED %, test  (threshold = val 1%)
                      <=510 tokens                 511-768                769-1024                  >1024                       ALL
first512  1.1 [0.7, 1.8] (16/1436)  0.3 [0.1, 1.2] (2/622)  1.1 [0.3, 4.0] (2/178)  1.8 [0.3, 9.6] (1/55)  0.9 [0.6, 1.4] (21/2291)
w384_lw   1.1 [0.7, 1.8] (16/1436)  0.5 [0.2, 1.4] (3/622)  0.6 [0.1, 3.1] (1/178)  0.0 [0.0, 6.5] (0/55)  0.9 [0.6, 1.3] (20/2291) 

HUMANS FLAGGED %, heldout_prompt  (threshold = val 1%)
                      <=510 tokens                   511-768                 769-1024                     >1024                        ALL
first512  3.6 [2.9, 4.6] (68/1880)  2.5 [1.8, 3.3] (42/1708)  6.5 [4.9, 8.6] (43/662)  7.9 [5.0, 12.4] (17/214)  3.8 [3.3, 4.4] (170/4464)
w384_lw   3.7 [3.0, 4.7] (70/1880)  2.8 [2.1, 3.7] (48/1708)  4.2 [2.9, 6.0] (28/662)   5.6 [3.2, 9.5] (12/214)  3.5 [3.0, 4.1] (158/4464) 



In [6]:
print('ALL VARIANTS, humans flagged %, test + held-out prompts together')
both = ev[ev['split'].isin(['test', 'heldout_prompt']) & (ev['label'] == 0)]
tab = pd.DataFrame({v: {b: cell(int(((both[v] >= THR[v]) & (both['bucket'] == b)).sum()), int((both['bucket'] == b).sum())) for b in both['bucket'].cat.categories}
                    | {'ALL': cell(int((both[v] >= THR[v]).sum()), len(both))} for v in VARIANTS}).T
print(tab.to_string())
print()
for split in ('test', 'heldout_prompt', 'heldout_generator'):
    print(f'AI CAUGHT %, {split}'); print(by_bucket(split, 1, show).to_string(), '\n')

ALL VARIANTS, humans flagged %, test + held-out prompts together
                       <=510 tokens                    511-768                  769-1024                       >1024                        ALL
first512   2.5 [2.1, 3.1] (84/3316)   1.9 [1.4, 2.5] (44/2330)   5.4 [4.0, 7.1] (45/840)    6.7 [4.3, 10.3] (18/269)  2.8 [2.5, 3.3] (191/6755)
w256_mean  2.6 [2.1, 3.2] (85/3316)   2.4 [1.8, 3.1] (55/2330)   4.0 [2.9, 5.6] (34/840)     4.5 [2.6, 7.6] (12/269)  2.8 [2.4, 3.2] (186/6755)
w256_max   1.2 [0.9, 1.7] (41/3316)   3.1 [2.5, 3.9] (72/2330)  9.0 [7.3, 11.2] (76/840)  16.4 [12.4, 21.2] (44/269)  3.4 [3.0, 3.9] (233/6755)
w256_lw    2.7 [2.2, 3.3] (88/3316)   2.3 [1.8, 3.0] (54/2330)   3.8 [2.7, 5.3] (32/840)     4.5 [2.6, 7.6] (12/269)  2.8 [2.4, 3.2] (186/6755)
w384_mean  2.6 [2.1, 3.2] (85/3316)   2.3 [1.8, 3.0] (54/2330)   3.9 [2.8, 5.5] (33/840)     4.5 [2.6, 7.6] (12/269)  2.7 [2.4, 3.1] (184/6755)
w384_max   0.8 [0.6, 1.2] (28/3316)  4.5 [3.7, 5.4] (105/2330)  8.1 [6.

                            <=510 tokens                        511-768                       769-1024 >1024                             ALL
first512  99.9 [99.7, 100.0] (2116/2118)  100.0 [99.1, 100.0] (429/429)  100.0 [97.0, 100.0] (124/124)     -  99.9 [99.7, 100.0] (2669/2671)
w384_lw   99.9 [99.7, 100.0] (2116/2118)  100.0 [99.1, 100.0] (429/429)  100.0 [97.0, 100.0] (124/124)     -  99.9 [99.7, 100.0] (2669/2671) 

AI CAUGHT %, heldout_generator
                           <=510 tokens                    511-768 769-1024 >1024                            ALL
first512  99.8 [99.6, 99.9] (2699/2704)  88.2 [65.7, 96.7] (15/17)        -     -  99.7 [99.5, 99.9] (2714/2721)
w384_lw   99.8 [99.6, 99.9] (2699/2704)  88.2 [65.7, 96.7] (15/17)        -     -  99.7 [99.5, 99.9] (2714/2721) 



### Paired view: which essays change verdict?

Both scorers see the same humans, so the interval on each rate overstates the uncertainty of the *difference*. Among the humans of `test` + held-out prompts, count the essays flagged by first-512 only, by the windowed score only, and by both. If windowing really fixes truncation false flags, the first count should dominate; an exact sign test on the changed essays gives the p-value.

In [7]:
rows = []
for v in VARIANTS[1:]:
    f0, f1 = both['first512'] >= THR['first512'], both[v] >= THR[v]
    only0, only1, both_ = int((f0 & ~f1).sum()), int((~f0 & f1).sum()), int((f0 & f1).sum())
    p = binomtest(only0, only0 + only1, 0.5).pvalue if only0 + only1 else 1.0
    rows.append({'variant': v, 'flagged by first-512 only': only0, 'flagged by windowed only': only1, 'flagged by both': both_,
                 'net change': only1 - only0, 'sign-test p': p})
pd.DataFrame(rows).set_index('variant').round(4)

,flagged by first-512 only,flagged by windowed only,flagged by both,net change,sign-test p
variant,,,,,
w256_mean,26,21,165,-5,0.5601
w256_max,62,104,129,42,0.0014
w256_lw,26,21,165,-5,0.5601
w384_mean,26,19,165,-7,0.3713
w384_max,87,128,104,41,0.0062
w384_lw,27,14,164,-13,0.0596


In [8]:
# AUC with intervals, first-512 vs the chosen variant (threshold-free; same essays)
rows = []
for split in ('test', 'heldout_prompt'):
    d = ev[ev['split'] == split]
    for v in ['first512', chosen]:
        a, lo, hi = auc_with_interval(d['label'], d[v], groups=d['dup_cluster'])
        rows.append({'split': split, 'scoring': v, 'AUC': f'{a:.4f} [{lo:.4f}, {hi:.4f}]'})
pd.DataFrame(rows)

,split,scoring,AUC
0,test,first512,"0.9995 [0.9987, 0.9999]"
1,test,w384_lw,"0.9993 [0.9984, 0.9999]"
2,heldout_prompt,first512,"0.9997 [0.9995, 0.9999]"
3,heldout_prompt,w384_lw,"0.9997 [0.9994, 0.9999]"


## 4 · Cost

`score_windows` runs the model once per window, so an essay of N tokens costs about `ceil((N − 510) / stride) + 1` forward passes. Timing for all 16,858 evaluation essays on the RTX 3050 (inference, fp16):

In [9]:
n = len(ev)
cost = pd.DataFrame({'seconds (all essays)': timing, 'ms per essay': 1000 * timing / n})
cost['x first-512'] = cost['seconds (all essays)'] / timing.iloc[0]
long = ev['n_tokens'] > 510
print(f"{long.mean():.1%} of essays are longer than 510 tokens; mean windows per essay: stride 256 {ev['n_windows_256'].mean():.2f}, stride 384 {ev['n_windows_384'].mean():.2f}; "
      f"per long essay: {ev.loc[long, 'n_windows_256'].mean():.2f} / {ev.loc[long, 'n_windows_384'].mean():.2f}")
cost.round(2)

29.8% of essays are longer than 510 tokens; mean windows per essay: stride 256 1.41, stride 384 1.34; per long essay: 2.36 / 2.13


,seconds (all essays),ms per essay,x first-512
first-512 (Detector.score),80.59,4.78,1.00
"windows, stride 256",104.59,6.20,1.30
"windows, stride 384",94.03,5.58,1.17


### Is the length effect just topic?

In first-512 scoring the false-flag rate climbs with essay length (2.5% for short essays, 5.4% and 6.7% above 768 tokens), but long essays also belong disproportionately to long-essay prompts, and notebook 15 showed that topic matters a lot. A within-prompt look: the flagged share by length bucket **inside each held-out prompt** (reported only, nothing is chosen from it), and the share of long essays per prompt.

In [10]:
hh = ev[(ev['split'] == 'heldout_prompt') & (ev['label'] == 0)].copy()
hh['long'] = np.where(hh['n_tokens'] > 510, 'long (>510 tokens)', 'short (<=510)')
rows = []
for prompt, g in hh.groupby('prompt_name'):
    for ln, gg in g.groupby('long'):
        rows.append({'prompt': prompt, 'length': ln, 'n': len(gg),
                     'first-512 flagged': cell(int((gg['first512'] >= THR['first512']).sum()), len(gg)),
                     f'{chosen} flagged': cell(int((gg[chosen] >= THR[chosen]).sum()), len(gg))})
print('share of long essays: ' + ', '.join(f"{p}: {100 * (g['n_tokens'] > 510).mean():.0f}%" for p, g in hh.groupby('prompt_name')))
pd.DataFrame(rows).set_index(['prompt', 'length'])

share of long essays: Does the electoral college work?: 46%, Summer projects: 77%


n          first-512 flagged            w384_lw flagged
prompt                           length                                                                        
Does the electoral college work? long (>510 tokens)  1243   0.8 [0.4, 1.5] (10/1243)    0.6 [0.3, 1.3] (8/1243)
                                 short (<=510)       1471   1.7 [1.2, 2.5] (25/1471)   1.8 [1.3, 2.7] (27/1471)
Summer projects                  long (>510 tokens)  1341   6.9 [5.6, 8.3] (92/1341)   6.0 [4.8, 7.4] (80/1341)
                                 short (<=510)        409  10.5 [7.9, 13.9] (43/409)  10.5 [7.9, 13.9] (43/409)

## Verdict: did the hypothesis hold?

**Mostly no.** Reading the whole essay does not fix the false flags in any way that the data can tell apart from noise.

- **Overall:** across the 6,755 humans of `test` and the held-out prompts, the chosen variant (stride 384, length-weighted mean) flags **178** against **191** for first-512: a net change of −13 essays (2.6% vs 2.8%). The paired sign test gives p = 0.06, and the changes in individual essays are of both signs (27 flagged only by first-512, 14 only by the windowed score). Not significant at the 5% level, and small: about 7% of the false flags.
- **Where it does help:** essays longer than 768 tokens, the ones truncation actually cuts most. Their flagged share falls from 5.4% and 6.7% to 3.5% and 4.5% (63 flagged → 41), but the intervals overlap, and the gain is partly paid back elsewhere: 511–768 tokens goes from 44 to 51 flagged and short essays 84 to 86 (the val-chosen threshold moves from 4.67 to 4.43, which also affects essays that were not changed at all).
- **"Max" aggregation is clearly worse:** 232 flagged against 191 (p = 0.006). One suspicious stretch is not evidence; long human essays contain such stretches.
- **Detection is unchanged:** AI essays caught on test, held-out prompts and held-out generators are identical for first-512 and the chosen variant (almost all AI essays are short enough to fit in one window), and AUC is unchanged (0.9997 [0.9994, 0.9999] either way on held-out prompts).
- **Within a prompt, length does not raise false flags at all.** Inside Electoral college the long essays are flagged 0.8% and the short ones 1.7%; inside Summer projects 6.9% and 10.5% (first-512 scoring). The climb of false flags with length in the pooled table comes from which prompts the long essays belong to (77% of Summer projects essays are long, 46% of Electoral college ones): topic composition, not truncation.
- **Cost:** 1.17× the scoring time of first-512 (5.6 ms against 4.8 ms per essay); 29.8% of essays need more than one window.
- **Caveat on the choice:** the variant was chosen on val by the pre-declared rule, but val could hardly tell the variants apart (99.50% vs 99.42% AI caught, a difference of one essay).

So truncation contributes at most a little, concentrated in very long essays, and the length-vs-false-flag pattern itself is a topic effect. The larger driver is topic (notebook 15, Summer projects vs Electoral college), which is what the next notebook (16b) diagnoses.

**Recommendation:** do not switch the scoring method. Keep first-512 for Phases 2b–4 so that all runs stay comparable. `score_windows` stays in `src/detector.py` for a possible later re-test with a better model.

## Key takeaways

- `Detector.score_windows` (new; `Detector.score` untouched) reproduces first-512 scores exactly for the 70% of essays that fit in one window.
- Windowed scoring changes false flags by about −7% (−13 of 191, p = 0.06): at best a small, partial fix, concentrated in essays over 768 tokens. Max aggregation makes things worse.
- No change in detection or AUC; 1.17× cost.
- Truncation is a minor contributor. Topic is the main one: go to 16b.